# 08. Wrap-up & Capstone Project: End-to-End Churn Pipeline
##### DSR Berlin | Data Science Fundamentals
##### Inspired by Adam Green (adgefficiency.com), Rachel Berryman, and Samson Afolabi

---

## 🎯 Capstone Objective
In this capstone notebook, you will bring together the complete **CRISP-DM lifecycle** covered across the entire course.
You will build a **production-ready, leak-free, reproducible machine learning pipeline** that ingests raw data, cleans and engineers features, trains models with cross-validation, and translates predictions into measurable business ROI.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import warnings

warnings.filterwarnings("ignore")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
RANDOM_SEED = 42

## Phase 1: Business Understanding & Project Brief

- **Client**: ConnectPlus Telecom GmbH.
- **Problem**: Customer churn is at ~26.5%, costing the business over €1.2M annually in lost subscription lifetime value.
- **Goal**: Build an automated classification model to identify churn risk 30 days in advance.
- **Intervention**: High-risk customers will receive a proactive retention package (cost: €25 per intervention, expected save rate: 40%).
- **Primary Metric**: **Recall & ROC-AUC** (catching maximum churners while preserving acceptable precision).


## Phase 2: Data Ingestion & The "Split First" Protocol

We load the raw dataset and **immediately isolate 20% into an untouched vault test set** before any statistical pre-processing.


In [ ]:
raw_df = pd.read_csv("../data/telco_customer_churn_raw.csv")
print(f"Raw Ingestion Shape: {raw_df.shape}")

# Deduplicate raw exact rows
raw_df = raw_df[~raw_df.duplicated()].copy()

# Standardize disguised missing values
raw_df["TotalCharges"] = pd.to_numeric(
    raw_df["TotalCharges"].replace(" ", np.nan), errors="coerce"
)
raw_df["MonthlyCharges"] = pd.to_numeric(raw_df["MonthlyCharges"], errors="coerce")
raw_df.loc[raw_df["MonthlyCharges"] < 0, "MonthlyCharges"] = np.nan
raw_df["PaymentMethod"] = raw_df["PaymentMethod"].replace(["?", "Unknown"], np.nan)

# Target vector
X = raw_df.drop(columns=["customerID", "Churn"])
y = (raw_df["Churn"] == "Yes").astype(int)

# Stratified Split (80% Train, 20% Test Vault)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)

print(f"Train Partition Shape: {X_train.shape}")
print(f"Test Vault Shape:      {X_test.shape}")
print(f"Train Base Churn Rate: {y_train.mean()*100:.2f}%")

## Phase 3: Building a Leak-Free `ColumnTransformer` Pipeline

Using Scikit-learn's `Pipeline` and `ColumnTransformer` guarantees that all imputation statistics, standard scalers, and one-hot encoders are fitted **strictly on the training folds**, with zero information leaking from test data.


In [ ]:
# Identify feature types
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [c for c in X_train.columns if c not in numeric_features]

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(
    f"Categorical features ({len(categorical_features)}): {categorical_features[:4]}..."
)

# 1. Numeric Transformation Sub-Pipeline
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# 2. Categorical Transformation Sub-Pipeline
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
        ),
    ]
)

# 3. Combine into Master ColumnTransformer Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## Phase 4: Progressive Model Comparison via 5-Fold Cross-Validation

We evaluate models across the progressive hierarchy:
- **Dummy Classifier** (Floor Benchmark)
- **Logistic Regression** (Interpretable Baseline)
- **Random Forest** (Non-linear Ensemble)


In [ ]:
models = {
    "1. Dummy Baseline": DummyClassifier(strategy="most_frequent"),
    "2. Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=RANDOM_SEED
    ),
    "3. Balanced Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=6, class_weight="balanced", random_state=RANDOM_SEED
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

results = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])
    roc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    results.append(
        {
            "Model": name,
            "Mean ROC-AUC": roc_scores.mean(),
            "Std ROC-AUC": roc_scores.std(),
        }
    )

results_df = pd.DataFrame(results)
print("5-Fold Cross-Validation Benchmark Results:")
print(results_df.round(4))

## Phase 5: Final Evaluation on Unseen Test Vault

We select the **Balanced Random Forest** as our production candidate, fit the complete pipeline on the full training partition, and evaluate on the untouched test vault.


In [ ]:
champion_pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=150,
                max_depth=6,
                class_weight="balanced",
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)

champion_pipe.fit(X_train, y_train)

y_pred_test = champion_pipe.predict(X_test)
y_prob_test = champion_pipe.predict_proba(X_test)[:, 1]

print("=== Classification Report on Test Vault ===")
print(classification_report(y_test, y_pred_test, target_names=["Retained", "Churn"]))
print(f"Test Vault ROC-AUC Score: {roc_auc_score(y_test, y_prob_test):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(cm, display_labels=["Retained", "Churn"])
disp.plot(ax=ax, cmap="Greens", colorbar=False)
ax.set_title("Test Vault Confusion Matrix (Champion Model)", fontweight="bold")
plt.tight_layout()
plt.show()

## Phase 6: Translating Predictions to Business Financial ROI

Let us calculate the net financial impact on ConnectPlus Telecom:
- Customer Lifetime Value (LTV): €1,200
- Intervention Cost: €25 per flagged customer
- Customer Success Save Rate: 40% (4 out of 10 targeted customers accept the offer and remain)


In [ ]:
tn, fp, fn, tp = cm.ravel()

# Total customers targeted = TP + FP
targeted_customers = tp + fp
intervention_cost = targeted_customers * 25

# Customers successfully saved = TP * 40%
saved_customers = tp * 0.40
revenue_preserved = saved_customers * 1200

net_financial_gain = revenue_preserved - intervention_cost

print(
    f"Total Churners Caught (TP):        {tp} of {tp + fn} ({tp/(tp+fn)*100:.1f}% Recall)"
)
print(f"Total Customers Targeted (TP+FP):  {targeted_customers}")
print(f"Total Campaign Cost (€25/cust):    €{intervention_cost:,.2f}")
print(f"Customers Successfully Retained:   {int(saved_customers)}")
print(f"Revenue Saved (€1,200/cust):       €{revenue_preserved:,.2f}")
print(f"--------------------------------------------------")
print(f"NET FINANCIAL ROI:                 €{net_financial_gain:,.2f}")

## 🎓 Independent Student Exercises

Now it's your turn! Try extending the pipeline to improve performance and business ROI:

### Exercise 1: Threshold Optimization
By default, the classifier predicts churn if $P(\text{Churn}) \ge 0.50$.
- Write a function that tests decision thresholds from $0.20$ to $0.80$ in steps of $0.05$.
- Compute the Net Financial ROI for each threshold.
- What threshold maximizes profit for the business?

### Exercise 2: Add Engineered Features to the Pipeline
In Notebook 04, we engineered features like `total_services` and `charge_tenure_ratio`.
- Wrap those transformations inside a custom `FunctionTransformer` or Scikit-learn `BaseEstimator`.
- Re-run cross-validation: does feature engineering lift the ROC-AUC score above 0.85?

### Exercise 3: Gradient Boosting Benchmark
- Import `HistGradientBoostingClassifier` (Scikit-learn's native tree-based booster).
- Train it within the pipeline: how does its recall compare to Random Forest?

---

### Appendix: Solution to Exercise 1 (Threshold Optimization)


In [ ]:
# Instructor Solution: Threshold vs Financial ROI
thresholds = np.linspace(0.20, 0.80, 25)
roi_curve = []

for thresh in thresholds:
    preds = (y_prob_test >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, preds)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()

    cost = (tp_t + fp_t) * 25
    revenue = (tp_t * 0.40) * 1200
    roi = revenue - cost
    roi_curve.append(roi)

best_idx = np.argmax(roi_curve)
best_thresh = thresholds[best_idx]
best_roi = roi_curve[best_idx]

plt.figure(figsize=(9, 4.5))
plt.plot(thresholds, roi_curve, color="navy", lw=2)
plt.axvline(
    best_thresh,
    color="red",
    linestyle="--",
    label=f"Optimal Threshold: {best_thresh:.2f} (ROI: €{best_roi:,.0f})",
)
plt.title("Net Financial ROI vs Decision Threshold", fontweight="bold")
plt.xlabel("Classification Probability Threshold")
plt.ylabel("Net Profit (€)")
plt.legend()
plt.tight_layout()
plt.show()

print(
    f"Optimal Decision Threshold: {best_thresh:.2f} yielding €{best_roi:,.2f} in net profit!"
)